In [ ]:
# Runtime setup for the Chapter 7 TensorFlow Quantum listings.
# The book snippets assume these small train/validation objects already exist.
import os
import sys
import time

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import numpy as np
import tensorflow as tf
import tf_keras

# TensorFlow Quantum 0.7.x expects the legacy Keras API.
# This also repairs notebook kernels where TensorFlow was already bound to Keras 3.
tf.keras = tf_keras

if "tensorflow_quantum" in sys.modules:
    import tensorflow_quantum as tfq
    pqc_base_module = tfq.layers.PQC.__mro__[1].__module__
    if not pqc_base_module.startswith("tf_keras"):
        raise RuntimeError(
            "TensorFlow Quantum was already imported with Keras 3. "
            "Restart the kernel, then run this notebook from the first cell."
        )

np.random.seed(7)
tf.random.set_seed(7)
tf.keras.utils.disable_interactive_logging()

x_train_scaled = np.array(
    [
        [0.05, 0.10],
        [0.20, 0.25],
        [0.35, 0.30],
        [2.30, 2.15],
        [2.55, 2.35],
        [2.80, 2.70],
    ],
    dtype=np.float32,
)
x_valid_scaled = np.array(
    [
        [0.15, 0.20],
        [0.40, 0.35],
        [2.40, 2.20],
        [2.70, 2.60],
    ],
    dtype=np.float32,
)
y_train = np.array([[0.0], [0.0], [0.0], [1.0], [1.0], [1.0]], dtype=np.float32)
y_valid = np.array([[0.0], [0.0], [1.0], [1.0]], dtype=np.float32)
input_dim = x_train_scaled.shape[1]


In [ ]:
import cirq
import sympy
import tensorflow as tf
import tensorflow_quantum as tfq

qubits = cirq.GridQubit.rect(1, 2)
theta = sympy.symbols("theta0:4")

def encode_example(x):
    """Map one two-feature example to a fixed data circuit."""
    circuit = cirq.Circuit()
    circuit += cirq.ry(float(x[0]))(qubits[0])
    circuit += cirq.ry(float(x[1]))(qubits[1])
    return circuit

def trainable_circuit():
    circuit = cirq.Circuit()
    circuit += cirq.ry(theta[0])(qubits[0])
    circuit += cirq.ry(theta[1])(qubits[1])
    circuit += cirq.CZ(qubits[0], qubits[1])
    circuit += cirq.rx(theta[2])(qubits[0])
    circuit += cirq.rx(theta[3])(qubits[1])
    return circuit

observables = [cirq.Z(qubits[0]), cirq.Z(qubits[1])]

quantum_inputs = tf.keras.Input(shape=(), dtype=tf.string, name="circuits")
quantum_features = tfq.layers.PQC(
    trainable_circuit(),
    observables,
    name="quantum_features",
)(quantum_inputs)

outputs = tf.keras.layers.Dense(1, activation="sigmoid")(quantum_features)
model = tf.keras.Model(inputs=quantum_inputs, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(0.02),
              loss="binary_crossentropy",
              metrics=["accuracy"])

# Build circuit tensors after the train/test split to avoid leakage.
# x_train_scaled should already contain angles on an appropriate range.
train_circuits = tfq.convert_to_tensor([encode_example(x) for x in x_train_scaled])
valid_circuits = tfq.convert_to_tensor([encode_example(x) for x in x_valid_scaled])
model.fit(train_circuits, y_train, batch_size=32, epochs=20,
          validation_data=(valid_circuits, y_valid))

In [ ]:
differentiator = tfq.differentiators.ParameterShift()

quantum_features = tfq.layers.PQC(
    trainable_circuit(),
    observables,
    differentiator=differentiator,
    name="quantum_features",
)(quantum_inputs)

experiment_card = {
    "quantum_layer": "tfq.layers.PQC",
    "differentiator": "ParameterShift",
    "num_qubits": len(qubits),
    "num_symbols": len(theta),
    "num_observables": len(observables),
    "shots": None,  # None here means exact simulator expectations.
}

In [ ]:
num_controls = len(theta)

feature_inputs = tf.keras.Input(shape=(input_dim,), name="features")
circuit_inputs = tf.keras.Input(shape=(), dtype=tf.string, name="circuits")

controller = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(num_controls),
], name="classical_controller")

symbol_values = controller(feature_inputs)

controlled_expectations = tfq.layers.ControlledPQC(
    trainable_circuit(),
    observables,
    name="controlled_quantum_layer",
)([circuit_inputs, symbol_values])

outputs = tf.keras.layers.Dense(1, activation="sigmoid")(
    controlled_expectations
)

hybrid_model = tf.keras.Model(
    inputs=[feature_inputs, circuit_inputs],
    outputs=outputs,
)
hybrid_model.compile(optimizer="adam",
                     loss="binary_crossentropy",
                     metrics=["accuracy"])

In [ ]:
# Support objects used by the later Chapter 7 snippets.
train_batch_size = len(y_train)
valid_batch_size = len(y_valid)

classical_train = tf.data.Dataset.from_tensor_slices(
    (x_train_scaled, y_train)
).batch(train_batch_size)
classical_valid = tf.data.Dataset.from_tensor_slices(
    (x_valid_scaled, y_valid)
).batch(valid_batch_size)
quantum_train = tf.data.Dataset.from_tensor_slices(
    (train_circuits, y_train)
).batch(train_batch_size)
quantum_valid = tf.data.Dataset.from_tensor_slices(
    (valid_circuits, y_valid)
).batch(valid_batch_size)
hybrid_train = tf.data.Dataset.from_tensor_slices(
    ((x_train_scaled, train_circuits), y_train)
).batch(train_batch_size)
hybrid_valid = tf.data.Dataset.from_tensor_slices(
    ((x_valid_scaled, valid_circuits), y_valid)
).batch(valid_batch_size)
train_data = quantum_train
valid_data = quantum_valid


def compile_binary_model(model, learning_rate=0.02):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_quantum_model(name="quantum_model", circuit=None, measured_observables=None):
    circuit = trainable_circuit() if circuit is None else circuit
    measured_observables = observables if measured_observables is None else measured_observables

    circuit_inputs = tf.keras.Input(shape=(), dtype=tf.string, name=f"{name}_circuits")
    quantum_features = tfq.layers.PQC(
        circuit,
        measured_observables,
        name=f"{name}_quantum_features",
    )(circuit_inputs)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(quantum_features)
    model = tf.keras.Model(inputs=circuit_inputs, outputs=outputs, name=name)
    return compile_binary_model(model)


def build_classical_model(name="classical_model", width=4):
    feature_inputs = tf.keras.Input(shape=(input_dim,), name=f"{name}_features")
    hidden = tf.keras.layers.Dense(width, activation="relu")(feature_inputs)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(hidden)
    model = tf.keras.Model(inputs=feature_inputs, outputs=outputs, name=name)
    return compile_binary_model(model)


def build_controlled_hybrid_model(name="hybrid_model"):
    feature_inputs = tf.keras.Input(shape=(input_dim,), name=f"{name}_features")
    circuit_inputs = tf.keras.Input(shape=(), dtype=tf.string, name=f"{name}_circuits")

    controller = tf.keras.Sequential(
        [
            tf.keras.layers.Dense(16, activation="relu"),
            tf.keras.layers.Dense(len(theta)),
        ],
        name=f"{name}_classical_controller",
    )
    symbol_values = controller(feature_inputs)
    controlled_expectations = tfq.layers.ControlledPQC(
        trainable_circuit(),
        observables,
        name=f"{name}_controlled_quantum_layer",
    )([circuit_inputs, symbol_values])
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(controlled_expectations)

    model = tf.keras.Model(
        inputs=[feature_inputs, circuit_inputs],
        outputs=outputs,
        name=name,
    )
    return compile_binary_model(model)


classical_model = build_classical_model("classical_only")
quantum_model = build_quantum_model("quantum_only")
hybrid_model = build_controlled_hybrid_model("hybrid")
matched_model = build_classical_model("matched_branch", width=len(observables))


def build_reuploading_circuit(qubits, depth):
    symbols = sympy.symbols(f"theta_d{depth}_0:{2 * len(qubits) * depth}")
    circuit = cirq.Circuit()
    symbol_index = 0

    for _ in range(depth):
        for qubit in qubits:
            circuit += cirq.ry(symbols[symbol_index])(qubit)
            symbol_index += 1
        if len(qubits) > 1:
            circuit += cirq.CZ(qubits[0], qubits[1])
        for qubit in qubits:
            circuit += cirq.rx(symbols[symbol_index])(qubit)
            symbol_index += 1

    return circuit, symbols


def build_tfq_model(circuit, symbols, observables):
    del symbols
    return build_quantum_model(
        name=f"depth_{len(list(circuit.all_operations()))}_model",
        circuit=circuit,
        measured_observables=observables,
    )


def train_with_diagnostics(model, train_data, valid_data):
    start = time.perf_counter()
    history = model.fit(train_data, epochs=1, verbose=0, validation_data=valid_data)
    seconds_per_epoch = time.perf_counter() - start

    inputs, labels = (
        next(iter(train_data))
        if isinstance(train_data, tf.data.Dataset)
        else train_data
    )
    loss_fn = tf.keras.losses.BinaryCrossentropy()
    with tf.GradientTape() as tape:
        predictions = model(inputs, training=True)
        loss = loss_fn(labels, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    gradient_norms = [
        float(tf.linalg.global_norm([gradient]).numpy())
        for gradient in gradients
        if gradient is not None
    ]

    diagnostics = {
        "median_quantum_grad_norm": (
            float(np.median(gradient_norms)) if gradient_norms else 0.0
        ),
        "seconds_per_epoch": seconds_per_epoch,
    }
    return history, diagnostics


In [ ]:
def train_and_evaluate(model, train_data, valid_data):
    history = model.fit(train_data, epochs=30, verbose=0,
                        validation_data=valid_data)
    return max(history.history["val_accuracy"])

scores = {
    "classical_only": train_and_evaluate(classical_model,
                                         classical_train,
                                         classical_valid),
    "quantum_only": train_and_evaluate(quantum_model,
                                       quantum_train,
                                       quantum_valid),
    "hybrid": train_and_evaluate(hybrid_model,
                                 hybrid_train,
                                 hybrid_valid),
    "matched_branch": train_and_evaluate(matched_model,
                                           classical_train,
                                           classical_valid),
}

for name, score in scores.items():
    print(f"{name:24s} validation accuracy = {score:.3f}")

In [ ]:
optimizer = tf.keras.optimizers.Adam(0.01)
loss_fn = tf.keras.losses.BinaryCrossentropy()

@tf.function
def train_step(inputs, labels):
    with tf.GradientTape() as tape:
        predictions = hybrid_model(inputs, training=True)
        loss = loss_fn(labels, predictions)

    gradients = tape.gradient(loss, hybrid_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, hybrid_model.trainable_variables))

    gradient_report = {}
    for variable, gradient in zip(hybrid_model.trainable_variables, gradients):
        if gradient is not None:
            gradient_report[variable.name] = tf.linalg.global_norm([gradient])

    return loss, gradient_report

In [ ]:
import pandas as pd

records = []

for depth in [1, 2, 3, 4]:
    circuit, symbols = build_reuploading_circuit(qubits, depth)
    model = build_tfq_model(circuit=circuit,
                            symbols=symbols,
                            observables=observables)
    history, diagnostics = train_with_diagnostics(model, train_data, valid_data)

    records.append({
        "depth": depth,
        "num_symbols": len(symbols),
        "num_operations": len(list(circuit.all_operations())),
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "final_train_loss": history.history["loss"][-1],
        "median_quantum_grad_norm": diagnostics["median_quantum_grad_norm"],
        "seconds_per_epoch": diagnostics["seconds_per_epoch"],
    })

depth_table = pd.DataFrame(records)
print(depth_table)